# Task 6: Real-time Video Colorization


In [6]:

import subprocess, sys, os

def ensure_numpy1():
    try:
        import numpy as np
        major = int(np.__version__.split('.')[0])
        if major >= 2:
            print(f"[!] Detected NumPy {np.__version__}. This version CAUSES CRASHES in OpenCV/Pandas.")
            print("[!] Downgrading to NumPy 1.26.4...")
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2.0', '--force-reinstall', '-q'])
            print("\n[SUCCESS] Downgrade complete. YOU MUST RESTART THE KERNEL NOW!")
            return False
        else:
            print(f"[OK] NumPy {np.__version__} is compatible.")
            return True
    except ImportError:
        print("[!] NumPy not found. Installing...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2.0', '-q'])
        return False

if not ensure_numpy1():

    raise Exception("ENVIRONMENT CHANGED: Please Restart your Jupyter Kernel to apply the NumPy fix.")


pkgs = ['gradio>=4.0.0', 'opencv-python-headless', 'scikit-image', 'tqdm', 'matplotlib']
print("Checking dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print("Done!")

[OK] NumPy 1.26.4 is compatible.
Checking dependencies...
Done!


In [7]:

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
import gradio as gr
import time, os, glob, random, gc
from PIL import Image
from tqdm import tqdm
from skimage import color
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 256

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

Device: cuda
GPU:    NVIDIA GeForce RTX 3050 4GB Laptop GPU


## Colorization Architectures
We use three models tailored for real-time performance:
1. **FastNet**: Shallow CNN for maximum FPS.
2. **DeepNet**: U-Net with skip connections for high-fidelity chrominance.
3. **VibrantNet**: Quality + post-processing saturation boost.

In [8]:


class FastNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 2, 3, padding=1), nn.Tanh()
        )
    def forward(self, x): return self.net(x)

class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
       
        def conv(ci, co, k=3, s=1, p=1): 
            return nn.Sequential(nn.Conv2d(ci, co, k, s, p), nn.BatchNorm2d(co), nn.LeakyReLU(0.2, True))
        def up(ci, co): 
            return nn.Sequential(nn.ConvTranspose2d(ci, co, 4, 2, 1), nn.BatchNorm2d(co), nn.ReLU(True))
        
        self.e1 = conv(1,   64, s=2)
        self.e2 = conv(64,  128, s=2)
        self.d1 = up(128, 64)
        self.d2 = up(128, 32)
        self.head = nn.Sequential(nn.Conv2d(32, 2, 3, padding=1), nn.Tanh())

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        d1 = self.d1(e2)
        d2 = self.d2(torch.cat([d1, e1], 1))
        return self.head(d2)

class VibrantNet(DeepNet):
    pass 

MODELS = {
    "FastNet (Speed)"   : FastNet(),
    "DeepNet (Quality)" : DeepNet(),
    "VibrantNet (Vivid)": VibrantNet()
}

for name, m in MODELS.items():
    MODELS[name] = m.to(DEVICE).eval()
    print(f"Loaded {name}")

Loaded FastNet (Speed)
Loaded DeepNet (Quality)
Loaded VibrantNet (Vivid)


In [9]:


def colorize_frame(frame_bgr, model_name, sat_mult=1.0):
    if frame_bgr is None: return None
    h, w = frame_bgr.shape[:2]

    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    lab = color.rgb2lab(img_resized.astype(np.float64) / 255.0)
    

    L = lab[:, :, 0]
    L_tensor = torch.FloatTensor(L / 50.0 - 1.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        model = MODELS.get(model_name, MODELS["DeepNet (Quality)"])
        ab_pred = model(L_tensor).squeeze().cpu().numpy().transpose(1, 2, 0) * 128.0

    boost = 1.3 if "VibrantNet" in model_name else 1.0
    ab_pred = ab_pred * boost * sat_mult

    result_lab = np.zeros((IMG_SIZE, IMG_SIZE, 3))
    result_lab[:, :, 0]  = L
    result_lab[:, :, 1:] = ab_pred

    result_rgb = (color.lab2rgb(result_lab) * 255).astype(np.uint8)
    result_rgb = cv2.resize(result_rgb, (w, h))
    return cv2.cvtColor(result_rgb, cv2.COLOR_RGB2BGR)

## Gradio  Interface
A full-featured dashboard for real-time experimentation.

In [ ]:


def handle_webcam(frame, model_name, intensity):
    if frame is None: return None
    # Gradio sends RGB, colorize_frame expects BGR
    bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    res_bgr = colorize_frame(bgr, model_name, intensity)
    return cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)

def handle_video(video_path, model_name, intensity):
    if not video_path: return None
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    out_path = "colorized_output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        colored = colorize_frame(frame, model_name, intensity)
        writer.write(colored)
    
    cap.release()
    writer.release()
    return out_path

with gr.Blocks(theme=gr.themes.Soft(), title="Task 6: Real-time Video Colorization") as ui:
    gr.Markdown("# Real-time Video Colorization")
    
    with gr.Row():
        with gr.Column(scale=1):
            model_choice = gr.Dropdown(choices=list(MODELS.keys()), value="DeepNet (Quality)", label="Model Select")
            sat_slider   = gr.Slider(0.5, 2.0, value=1.0, label="Color Intensity")
        
    with gr.Tabs():
        with gr.Tab("Live Webcam"):
            with gr.Row():
                cam_in  = gr.Image(sources="webcam", streaming=True, label="Input")
                cam_out = gr.Image(label="Colorized")
            cam_in.stream(fn=handle_webcam, inputs=[cam_in, model_choice, sat_slider], outputs=cam_out, stream_every=0.1)
            
        with gr.Tab("Video Upload"):
            with gr.Row():
                vid_in  = gr.Video(label="Source")
                vid_out = gr.Video(label="Colorized Output")
            btn = gr.Button("Colorize Video", variant="primary")
            btn.click(fn=handle_video, inputs=[vid_in, model_choice, sat_slider], outputs=vid_out)

print("Opening GUI...")
ui.launch(server_port=9789, share=False)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_25392\698330496.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Task 6: Real-time Video Colorization") as ui:


Opening GUI...


OSError: Cannot find empty port in range: 7861-7861. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.